<a href="https://colab.research.google.com/github/shashank3110/genAI/blob/main/langraph_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -U langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4


In [16]:
pip install -U langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00


In [74]:
from langgraph.graph import StateGraph, MessagesState, START, END


In [75]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

In [76]:
import os
os.environ["GOOGLE_API_KEY"]=""

In [77]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # or gemini-1.5-flash
    temperature=0
)

In [78]:
# Define tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b

@tool
def power(a: int, b: int) -> float:
    """Calculates exponents: a to power b

    Args:
        a: First int
        b: Second int
    """

    return a**b



In [79]:
tools = [add, multiply, divide, power]
tool_names = [tool.name for tool in tools]
llm_with_tools = llm.bind_tools(tools)

In [145]:
from langgraph.graph import StateGraph
from typing import TypedDict, List
from langchain_core.messages import BaseMessage

# class AgentState(TypedDict):
#     messages: List[BaseMessage]


class AgentState(MessagesState):
    pass



In [146]:
from langchain_core.messages import AIMessage

def agent(state: AgentState):
    if not state["messages"]:
        raise ValueError("Empty state!")

    response = llm_with_tools.invoke(state["messages"])
    print(f"[DEBUG] MODEL OUTPUT: {response}")
    print("[DEBUG] TOOL CALLS:", getattr(response, "tool_calls", None))

    return {"messages":  state["messages"] + [response]}


In [147]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)


In [148]:
# for conditional branching/routing


def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    return "end"


In [149]:
#DEBUG CODE
def tool_node_debug(state):
    print("\n🔥 TOOL NODE INPUT:", state)

    result = tool_node.invoke(state)

    print("\n🔥 TOOL NODE OUTPUT:", result)

    return result

In [150]:
# DEBIUG CODE
def debug_node(name, fn):
    def wrapper(state):
        print(f"\n🔥 {name} INPUT:")
        for m in state["messages"]:
            print(type(m), m)

        result = fn(state)

        print(f"\n🔥 {name} OUTPUT:")
        for m in result.get("messages", []):
            print(type(m), m)

        return result
    return wrapper


In [156]:
math_graph = StateGraph(AgentState)
math_graph.add_node("agent", agent)
math_graph.add_node("tools",tool_node)
# math_graph.add_node("tools",tool_node_debug)

# math_graph.add_node("agent", debug_node("AGENT", agent))
# math_graph.add_node("tools", debug_node("TOOLS", tool_node_debug))


math_graph.set_entry_point("agent")

math_graph.add_conditional_edges("agent", should_continue,  {
        "tools": "tools",
        "end": "__end__"
    }
    )

math_graph.add_edge("tools", "agent")
# math_graph.add_edge("tools", "end")

In [157]:
math_agent = math_graph.compile()

In [153]:
from langchain_core.messages import HumanMessage, SystemMessage


In [158]:
result = math_agent.invoke(

     {
          "messages": [
              SystemMessage(content="You are a math assistant. Always use tools for calculations."),
              HumanMessage(content="What is 3 * 5?"),
              HumanMessage(content="What is 389 + 15?"),
              HumanMessage(content="What is 89 - 15?"),
              HumanMessage(content="What is 8 to power 2?"),
              HumanMessage(content=" 18 /2?")

              ]
     }

)

[DEBUG] MODEL OUTPUT: content='' additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 5}'}, '__gemini_function_call_thought_signatures__': {'7ef79fe6-551b-4491-968d-a9e5fa8bbabd': 'CvUDAb4+9vuvLDbfMhWyfA8rsU6r3pRCrEvvh4fdUN6ShvoI1rOrblU8/tqpeZJhVZIcxVKgTf9inm7yW6qeSPL/YE80MXm5Bv7o4PMMk+U0gR1iV6oRYajO0hTD9i9j1Id4yzR3pGZZhP8OruZbk5zMDZkhsVgKu3WUWUSD6ZNf8q1qwJYG6UBtvUMoFoYbG5IDDI7W75xHsisEk85Ud3uZQ25Gu1t3dUMj6tkjUtbVpgXQjS9egbCiJdHBf4cmzTUiIVj0tPDpV1AyORm+agO7FgowCKyxXXZQKJ7sLd42jpBE6rbVcjtv8D5BZTLsLaCWK1qQbxqTqQwijN35fLmXsXBuNDAPnCAc7YEuJIrhKFE5j3Wkv2tSU+G6kGSQFhr6k8oZcS23/GoZ4GkGuiC9tPyP1TWrpOF7+PupHDVvFTtRTVThCKuamjsKwe87Nxq2KnnSFxkAixmOfVT/4faU6rmmQv35hf5eItKm1oqh4W0b2HI2MAMcklsLLwI0Xmeg9UFdFy374L9qDUraqW+QSfwVnf7lit7G9awZWhmsm2RWCzg3d9YfGYozcFFRNiUt2HbEYskZwqyA7BLW1jO1SYfS4UrA2Wr009HZckVArwf/eZsOKKb+X2eRSbE2Ed4qZKbzYWkrHOE7C4gdo7ccOxd8Gu64'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_prov

In [159]:
print(result["messages"][-1].content)

[{'type': 'text', 'text': "3 * 5 = 15\n389 + 15 = 404\nI can't help you with 89 - 15, as I don't have a subtraction tool.\n8 to power 2 = 64\n18 / 2 = 9", 'extras': {'signature': 'CvgCAb4+9vtcqri0giA9QxGphEPjQSN/vv23u+u7mPBfY610A8dtJpHi+BgElZeN4WV1ZiukjwlCnXnD+G/LwEH6Yl+DOQswHhrOasx8rv8wBGQmaTIHlupyM77tNckW9FupPAMajLjcP2h8yt+nlN64RKzjYoaZkYXUDdO6Ie/IfhSrQqyMmCeFyLLW/ceL1aJJX6jVFe0V8gYQFekAQ5yXJDYoVpIxcim0q/5H3mBQQb8Ta1GkJAezc3A90lCetICP9muKVT+QOAOvpKmUGjiT5F+tpOTGiqRxg0Bd95J+mLAsAMqudG7Jt0EbopIkR9NNSBZ8FI17/Fdj2HtuU/oSHg+q8CxrnyU9wLfzIVDUh3esP2rcfC2w0mFwPDF5y3oOkh8SN78FgjJdoZyx2D1XShIoQ/oeLR/SY7aV6xpZV2x1vL04T7L1e906Wd3BPtZxwMeBMxYeZbCkwrGkGCNa0FdrO8KZwwvk4ARPb1JnBbyjxmUnvVW3Jg=='}}]


In [161]:
# result